## Email Ingestion - Underwriting

In [1]:
import os
import glob
import json
from pathlib import Path
import pandas as pd
from groq import Groq
from dotenv import load_dotenv

# Load your Groq key
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL_NAME = "llama-3.3-70b-versatile"


def default_data_directory():
    """Find the interview-pack sample-email directory from the notebook folder."""
    candidates = [
        Path.cwd() / "sample emails",
        Path.cwd().parent / "interview_pack" / "interview_pack" / "sample emails",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    searched = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Could not find the sample emails directory. Searched:\n{searched}")


def group_submission_files(directory=None):
    """Group all Markdown email and attachment files by submission ID."""
    directory = Path(directory) if directory else default_data_directory()
    filepaths = sorted(directory.glob("*.md"))
    if not filepaths:
        raise FileNotFoundError(f"No Markdown files found in {directory}")

    submissions = {}
    for path in filepaths:
        filename = path.name
        sub_id = filename.split("_")[0]
        content = path.read_text(encoding="utf-8")
        submissions.setdefault(sub_id, "")
        submissions[sub_id] += f"\n\n--- FILE: {filename} ---\n{content}"

    return submissions


def extract_submission_data(sub_id, text_payload):
    """Pass the combined email and attachments to Groq for extraction."""
    system_prompt = (
        "You are an insurance underwriting AI. Extract the exact fields from the provided broker submissions. "
        "Return ONLY a raw, valid JSON object with no markdown formatting or backticks. "
        "Schema:\n"
        "{\n"
        '  "company_name": "string",\n'
        '  "revenue": integer (numeric only, no symbols, convert from text if needed),\n'
        '  "countries": ["list", "of", "strings"],\n'
        '  "industry": "string",\n'
        '  "requested_coverages": ["list", "of", "strings"]\n'
        "}"
    )

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=0.0,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Extract data from these files:\n{text_payload}"}
            ]
        )

        raw_output = response.choices[0].message.content.strip()
        if raw_output.startswith("```json"):
            raw_output = raw_output[7:-3].strip()

        data = json.loads(raw_output)
        data["submission_id"] = sub_id
        return data

    except Exception as e:
        print(f"Failed on {sub_id}: {e}")
        return {"submission_id": sub_id, "error": str(e)}


In [2]:
# Select a model available to the configured Groq key
MODEL_NAME = "openai/gpt-oss-20b"
print(f"Using Groq model: {MODEL_NAME}")

Using Groq model: openai/gpt-oss-20b


In [ ]:
# Ingest and group files
submission_docs = group_submission_files()
print(f"Loaded {len(submission_docs)} submissions from {default_data_directory()}")

Loaded 50 submissions from c:\Users\beckk\Documents\Python\Interview\interview_pack\interview_pack\sample emails


In [ ]:
# Run extraction pipeline
import time

extracted_records = []
for sub_id, payload in submission_docs.items():
    print(f"Processing {sub_id}...")
    record = extract_submission_data(sub_id, payload)
    extracted_records.append(record)
    
    # 20-second delay guarantees < 8000 tokens per minute
    time.sleep(20)

Processing E001...
Processing E002...
Processing E003...
Processing E004...
Processing E005...
Processing E006...
Processing E007...
Processing E008...
Processing E009...
Processing E010...
Processing E011...
Processing E012...
Processing E013...
Processing E014...
Processing E015...
Processing E016...
Processing E017...
Processing E018...
Processing E019...
Processing E020...
Processing E021...
Processing E022...
Processing E023...
Processing E024...
Processing E025...
Processing E026...
Processing E027...
Processing E028...
Processing E029...
Processing E030...
Processing E031...
Processing E032...
Processing E033...
Processing E034...
Processing E035...
Processing E036...
Processing E037...
Processing E038...
Processing E039...
Processing E040...
Processing E041...
Processing E042...
Processing E043...
Processing E044...
Processing E045...
Processing E046...
Processing E047...
Processing E048...
Processing E049...
Processing E050...


In [ ]:

# Format to target dataframe
df = pd.DataFrame(extracted_records)

# Reorder columns to match the ground truth format
cols = ["submission_id", "company_name", "revenue", "countries", "industry", "requested_coverages"]
df = df[[c for c in cols if c in df.columns]]

print("Extraction complete.")
display(df.head())

# Save to CSV for the evaluation phase
df.to_csv("llm_extracted_submissions.csv", index=False)

Extraction complete.


,submission_id,company_name,revenue,countries,industry,requested_coverages
0,E001,Everstead Pharma Ltd,14608818,"[Sweden, Canada, Singapore]",Manufacturing,"[Property, Kidnap & Ransom]"
1,E002,Crestline Textiles Ltd,38122777,"[Spain, Norway, Belgium]",Technology,"[General Liability, Property]"
2,E003,Granite Analytics Ltd,7808428,[United States],Healthcare,"[Professional Indemnity, Technology E&O, Direc..."
3,E004,Highland Logistics Ltd,35958262,"[United Arab Emirates, United Kingdom]",Fintech,"[Professional Indemnity, Management Liability,..."
4,E005,Ironclad Trading Ltd,29626602,[Denmark],Retail,[Management Liability]
